# Modul 6: CNN Lanjutan dan Transfer Learning

**Nama:** ISI NAMA  
**NIM:** ISI NIM  
**Kelas:** ISI KELAS  
**Tanggal:** YYYY-MM-DD  
**Sumber dataset:** ISI (nama dataset dan asalnya)

Simpan berkas ini sebagai `M06_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Letakkan dataset di `data/raw/transfer/` dengan satu subfolder per kelas, minimal 4 kelas dan 120 citra per kelas.
3. Transformasi masukan **wajib** berasal dari `weights.transforms()`.
4. Augmentasi hanya boleh melekat pada split latih.
5. Pembekuan wajib dibuktikan dengan **dua** cara.
6. Test set dipakai **satu kali** pada bagian G.
7. Luaran: `M06_NIM.ipynb`, `M06_NIM.pdf`, `M06_NIM_metrics.csv`.

In [ ]:
import copy
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from torchvision.models import ResNet18_Weights, resnet18

NIM = 'TODO'                 # contoh: '120450123'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'torch': torch.__version__, 'device': str(DEVICE), 'seed': SEED})

## A. Pre-lab - 10 poin

1. **Mengapa citra harus dinormalisasi dengan statistik yang sama seperti saat pralatih:** TODO
2. **Apa yang terjadi pada gradien layer beku, dan pada waktu komputasinya:** TODO
3. **Jumlah parameter `Linear(512, 5)` termasuk bias:** TODO
4. **Mengapa fine-tuning memakai learning rate jauh lebih kecil daripada from scratch:** TODO

## B. Bobot pralatih dan praproses - 15 poin

Ambil transformasi dari objek bobot, lalu bangun augmentasi yang memakai normalisasi **yang sama**.

In [ ]:
bobot = ResNet18_Weights.IMAGENET1K_V1

# TODO 1: ambil transformasi evaluasi dari objek bobot, cetak isinya,
#         lalu baca MEAN, STD, dan ukuran crop darinya.
tf_eval = ...
print(tf_eval)
MEAN, STD = ..., ...
SISI = ...

# TODO 2: susun augmentasi ringan: RandomResizedCrop(SISI, scale=(0.8, 1.0)),
#         RandomHorizontalFlip, ToTensor, lalu Normalize dengan MEAN dan STD
#         YANG SAMA seperti di atas.
tf_aug = ...

print('crop:', SISI, '| mean:', MEAN, '| std:', STD)
assert list(MEAN) == [0.485, 0.456, 0.406], 'MEAN harus berasal dari bobot pralatih'

In [ ]:
ROOT = Path('../../data/raw/transfer')
if not ROOT.exists():
    ROOT = Path('data/raw/transfer')

ds_aug = ImageFolder(ROOT, transform=tf_aug)      # untuk split latih
ds_polos = ImageFolder(ROOT, transform=tf_eval)   # untuk validasi dan uji
KELAS = ds_polos.classes
y = np.array(ds_polos.targets)

# TODO 3: split terstratifikasi 70% latih, 15% validasi, 15% uji (random_state=SEED).
idx_latih, idx_sisa = ...
idx_val, idx_uji = ...

ds_latih = Subset(ds_aug, idx_latih)
ds_val = Subset(ds_polos, idx_val)
ds_uji = Subset(ds_polos, idx_uji)

print(f'kelas ({len(KELAS)}): {KELAS}')
print(f'latih {len(ds_latih)}  validasi {len(ds_val)}  uji {len(ds_uji)}')
print('citra per kelas:', np.bincount(y).tolist())

assert len(KELAS) >= 4, 'dataset harus punya minimal empat kelas'
assert np.bincount(y).min() >= 120, 'setiap kelas minimal 120 citra'
assert ds_val.dataset is ds_polos and ds_uji.dataset is ds_polos, \
    'validasi dan uji tidak boleh memakai dataset beraugmentasi'
print('dataset dan split sesuai protokol')

## C. Head baru dan hitungan parameter - 15 poin

In [ ]:
K = len(KELAS)

def buat_model(pralatih=True):
    """TODO 4: muat resnet18 (pralatih atau acak), ganti model.fc dengan
    nn.Linear(in_features, K), lalu pindahkan ke DEVICE.
    Panggil seed_everything(SEED) lebih dahulu."""
    raise NotImplementedError

model = buat_model()
total = sum(p.numel() for p in model.parameters())
head = sum(p.numel() for p in model.fc.parameters())
layer4 = sum(p.numel() for p in model.layer4.parameters())

print(f'total          : {total:,}')
print(f'head           : {head:,}  ({100*head/total:.3f}%)')
print(f'layer4 + head  : {layer4 + head:,}  ({100*(layer4+head)/total:.1f}%)')
assert head == 512 * K + K, 'dimensi head belum sesuai jumlah kelas'

**Letak parameter.** Berapa persen parameter model yang terbuka bila hanya `layer4` dan head dilatih, dan apa artinya bagi klaim "partial fine-tuning itu ringan"? TODO

## D. Membekukan dan membuktikannya - 15 poin

Dua bukti wajib: status `requires_grad`, **dan** bobot beku yang tidak bergerak setelah satu update.

In [ ]:
def bekukan(model, strategi):
    """TODO 5: 'frozen' -> hanya fc; 'partial' -> layer4 dan fc;
    'full'/'scratch' -> seluruhnya. Kembalikan model."""
    raise NotImplementedError

def n_terlatih(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for strategi in ('frozen', 'partial', 'full'):
    m = bekukan(buat_model(), strategi)
    print(f'{strategi:>8}: {n_terlatih(m):>11,} ({100*n_terlatih(m)/total:5.1f}%)')

m_cek = bekukan(buat_model(), 'frozen')
assert n_terlatih(m_cek) == head, 'strategi frozen harus melatih head saja'

In [ ]:
# TODO 6: bukti kedua. Simpan salinan m_cek.layer1[0].conv1.weight,
#         buat optimizer HANYA dari parameter dengan requires_grad=True,
#         jalankan satu langkah update pada satu batch, lalu bandingkan.
salinan = ...
raise NotImplementedError

selisih = (m_cek.layer1[0].conv1.weight.detach() - salinan).abs().max().item()
print(f'selisih maksimum bobot beku: {selisih:.2e}')
assert selisih == 0.0, 'bobot beku ternyata berubah — periksa isi optimizer'
print('pembekuan terbukti')

**Mengapa dua bukti?** Jelaskan keadaan ketika `requires_grad=False` sudah dipasang tetapi bobot tetap bergerak: TODO

## E. Empat run terkendali - 25 poin

Data, seed, batch, dan epoch sama. Yang berbeda hanya bobot awal dan bagian yang boleh berubah.

In [ ]:
BATCH, EPOCH = 32, 5

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds):
    """TODO 7: kembalikan (loss rata-rata, akurasi). Jangan lupa model.eval()."""
    raise NotImplementedError

def jalankan(strategi, lr, label, epoch=EPOCH):
    """TODO 8: satu fungsi pelatihan untuk SELURUH run.

    - buat_model(pralatih = strategi != 'scratch'), lalu bekukan sesuai strategi;
    - optimizer HANYA menerima parameter dengan requires_grad=True;
    - kembalikan (model, riwayat, catatan) dengan catatan memuat run_id, seed,
      strategi, bobot_awal, learning_rate, parameter_total, parameter_terlatih,
      augmentasi, train_loss, val_loss, val_acc, gap, detik_per_epoch.
    """
    raise NotImplementedError

hasil, kurva, model_simpan = [], {}, {}
for strategi, lr, label in [('scratch', 1e-3, 'from-scratch'),
                            ('frozen', 1e-3, 'frozen-backbone'),
                            ('partial', 1e-4, 'partial-layer4'),
                            ('full', 1e-4, 'full-finetune')]:
    m, r, catatan = jalankan(strategi, lr, label)
    hasil.append(catatan); kurva[label] = r; model_simpan[label] = m

print(pd.DataFrame(hasil)[['run_id', 'bobot_awal', 'parameter_terlatih',
                           'val_loss', 'val_acc', 'gap',
                           'detik_per_epoch']].to_string(index=False))
assert len(hasil) == 4, 'harus ada empat run strategi'

In [ ]:
# TODO 9: tampilkan kurva validation accuracy keempat run dalam satu grafik berlabel.
raise NotImplementedError

**Checkpoint menit ke-80.** Tunjukkan kepada asisten: parameter terlatih ketiga strategi, transformasi yang berasal dari objek bobot, dan selisih nol pada bobot beku.

## F. Augmentasi - bagian dari 20 poin

Bandingkan tanpa augmentasi dan augmentasi agresif pada strategi terbaik Anda.

In [ ]:
# TODO 10: susun tf_keras (RandomResizedCrop scale=(0.3,1.0), RandomRotation(45),
#          flip, ToTensor, Normalize yang SAMA), lalu jalankan dua run tambahan
#          pada strategi terbaik: augmentasi 'tanpa' dan 'agresif'.
#          Petunjuk: ubah ds_aug.transform sebelum tiap run, dan kembalikan
#          ke tf_aug setelah selesai.
raise NotImplementedError

tabel = pd.DataFrame(hasil)
print(tabel[['run_id', 'strategi', 'augmentasi', 'val_loss', 'val_acc',
             'gap']].to_string(index=False))
assert len(tabel) == 6, 'harus ada empat run strategi + dua run augmentasi'

**Dampak augmentasi.** Ke arah mana akurasi bergerak, dan pada jenis citra seperti apa augmentasi agresif dapat menghapus ciri penentu kelas? TODO

## G. Evaluasi test satu kali - bagian dari 20 poin

In [ ]:
# TODO 11: pilih strategi terbaik dari validation loss, evaluasi ds_uji SATU KALI,
#          tampilkan confusion matrix, lalu cetak lima prediksi salah beserta
#          label benar dan label prediksinya.
raise NotImplementedError

**Analisis lima kesalahan.**

| No | Label benar | Prediksi | Dugaan penyebab |
|----|-------------|----------|-----------------|
| 1 | TODO | TODO | TODO |
| 2 | TODO | TODO | TODO |
| 3 | TODO | TODO | TODO |
| 4 | TODO | TODO | TODO |
| 5 | TODO | TODO | TODO |

In [ ]:
# TODO 12: simpan seluruh run ke metrics.csv.
tabel.insert(0, 'module', 'M06')
tabel.insert(1, 'student_id', NIM)
tabel.to_csv(f'M06_{NIM}_metrics.csv', index=False)
print(f'{len(tabel)} baris tersimpan')

## H. Pertanyaan analisis - bagian dari 20 poin

1. Berapa selisih accuracy from scratch dan frozen backbone, dan berapa perbandingan parameter terlatihnya? TODO
2. Apakah full fine-tuning selalu mengalahkan partial? Sertakan waktu per epoch. TODO
3. Mengapa membuka `layer4` saja sudah membuka sekitar tiga perempat parameter? TODO
4. Apa yang berubah ketika augmentasi ditambahkan, dan pada strategi mana paling terasa? TODO
5. Strategi mana untuk laptop tanpa GPU dengan data di bawah 500 citra? **Dukung dengan tiga angka**: akurasi, waktu per epoch, dan parameter terlatih. TODO

## Checklist sebelum mengumpulkan

- [ ] Identitas, seed, device, dan sumber dataset tercantum.
- [ ] Transformasi berasal dari `weights.transforms()` dan isinya tercetak.
- [ ] Augmentasi hanya melekat pada split latih.
- [ ] Bukti pembekuan lengkap: `requires_grad` dan selisih bobot nol.
- [ ] Parameter terlatih dilaporkan untuk keempat run.
- [ ] Enam run tercatat di `metrics.csv`.
- [ ] Test set dipakai satu kali pada bagian G.
- [ ] Rekomendasi akhir menyebut akurasi, waktu, dan parameter terlatih.
- [ ] Notebook lolos *Restart Kernel and Run All*.